# 02 — Embeddings & semantic clustering

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")


True

In [2]:

from embeddings.clusterer import SemanticClusterer
from embeddings.embed_generator import TEIEmbedder
from embeddings.pattern_detector import PatternDetector



In [3]:
PARQUET_IN = ROOT / "data" / "processed" / "transactions_clean.parquet"
PROCESSED = ROOT / "data" / "processed"
VIZ = ROOT / "visualizations"
VIZ.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("Parquet exists:", PARQUET_IN.is_file())

ROOT: /mnt/data-disk/FinSight/FinSight-AI
Parquet exists: True


In [4]:
if not PARQUET_IN.is_file():
    raise FileNotFoundError(f"Run 01_preprocessing first. Missing {PARQUET_IN}")

df_full = pd.read_parquet(PARQUET_IN)
SAMPLE_N = min(10_000, len(df_full))
df_sample = df_full.sample(n=SAMPLE_N, random_state=42).reset_index(drop=True)
print("full rows:", len(df_full), "| embed sample:", len(df_sample))

full rows: 500000 | embed sample: 10000


In [5]:
TEI_URL = (
    os.getenv("EMBEDDING_SERVER_URL")
    or os.getenv("TEI_BASE_URL")
    or "http://localhost:8001"
).rstrip("/")
embedder = TEIEmbedder(base_url=TEI_URL, embed_batch_size=32)
embeddings = embedder.embed_transactions(df_sample)
print("TEI:", TEI_URL)
print("embedding matrix:", embeddings.shape, embeddings.dtype)

TEI: http://localhost:8001
embedding matrix: (10000, 1024) float32


In [6]:
clusterer = SemanticClusterer(n_clusters=6)
elbow_path = VIZ / "elbow_curve.png"
optimal_k = clusterer.find_optimal_k(embeddings, k_range=range(3, 10), plot_path=elbow_path)
print("optimal_k (heuristic):", optimal_k, "| plot:", elbow_path)


optimal_k (heuristic): 4 | plot: /mnt/data-disk/FinSight/FinSight-AI/visualizations/elbow_curve.png


In [7]:
clusterer.n_clusters = optimal_k
labels = clusterer.fit_predict(embeddings)
df_sample = df_sample.copy()
df_sample["cluster"] = labels
summary = clusterer.label_clusters(df_sample, labels)
for cid, info in sorted(summary.items()):
    print(cid, info["label"], "|", info["stats"])

0 Bills & Purchases | {'size': 2340, 'mean_amount': 15054.599572649573, 'top_category': 'Bills & Purchases'}
1 Withdrawal | {'size': 3622, 'mean_amount': 184466.43290999447, 'top_category': 'Withdrawal'}
2 Deposit | {'size': 3060, 'mean_amount': 324749.73623202613, 'top_category': 'Deposit'}
3 Bills & Purchases (Small) | {'size': 978, 'mean_amount': 2404.6673926380367, 'top_category': 'Bills & Purchases'}


In [8]:
import importlib
import os

import embeddings.clusterer as clusterer_mod

importlib.reload(clusterer_mod)
clusterer = clusterer_mod.SemanticClusterer(n_clusters=clusterer.n_clusters)

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333").rstrip("/")
QDRANT_COLLECTION = os.getenv("QDRANT_COLLECTION", "transactions_embeddings")

qdrant, _ = clusterer.build_qdrant_index(
    embeddings,
    qdrant_url=QDRANT_URL,
    collection_name=QDRANT_COLLECTION,
    upsert_batch_size=64,
)

print("Qdrant URL:", QDRANT_URL)
print("Qdrant collection:", QDRANT_COLLECTION)
print("Qdrant indexed points:", qdrant.count(collection_name=QDRANT_COLLECTION).count)

Qdrant URL: http://localhost:6333
Qdrant collection: transactions_embeddings
Qdrant indexed points: 10000


In [9]:
detector = PatternDetector()
patterns = {
    "recurring_payments": detector.detect_recurring_payments(df_full),
    "spending_trends": detector.detect_spending_trends(df_full),
    "behavioral_patterns": detector.detect_behavioral_patterns(df_full),
}
print("behavioral_patterns:")
for line in patterns["behavioral_patterns"]:
    print(" -", line)

behavioral_patterns:
 - Weekday transaction volume exceeds weekend volume.
 - Several days show many small transactions (< $100), suggesting rapid micro-activity.


In [11]:
out_parquet = PROCESSED / "transactions_clustered.parquet"
out_clusters = PROCESSED / "cluster_labels.json"
out_patterns = PROCESSED / "patterns.json"

In [12]:
df_sample.to_parquet(out_parquet, index=False)
with open(out_clusters, "w", encoding="utf-8") as f:
    json.dump(
        {
            "optimal_k": optimal_k,
            "sample_size": SAMPLE_N,
            "elbow_plot": str(elbow_path.relative_to(ROOT)),
            "clusters": {str(k): v for k, v in summary.items()},
            "vector_store": {
                "provider": "qdrant",
                "url": QDRANT_URL,
                "collection": QDRANT_COLLECTION,
            },
        },
        f,
        indent=2,
        default=str,
    )
with open(out_patterns, "w", encoding="utf-8") as f:
    json.dump(patterns, f, indent=2, default=str)

print("Wrote:", out_parquet)
print("Wrote:", out_clusters)
print("Wrote:", out_patterns)

Wrote: /mnt/data-disk/FinSight/FinSight-AI/data/processed/transactions_clustered.parquet
Wrote: /mnt/data-disk/FinSight/FinSight-AI/data/processed/cluster_labels.json
Wrote: /mnt/data-disk/FinSight/FinSight-AI/data/processed/patterns.json


In [13]:
query = "Withdrawal of $500.00 on 2024-01-05 (Friday), Medium transaction"
query_vector = embedder.embed_text(query)
hits = qdrant.search(
    collection_name=QDRANT_COLLECTION,
    query_vector=query_vector,
    limit=5,
)

hit_ids = [int(hit.id) for hit in hits]
neighbors = df_sample.iloc[hit_ids].copy() if hit_ids else df_sample.iloc[[]].copy()
neighbors[["amount", "category", "date", "cluster"]] if len(neighbors) else neighbors

,amount,category,date,cluster
5161,2035.00,Withdrawal,2024-01-01,1
8743,1457.00,Withdrawal,2024-01-01,1
9704,3081.73,Withdrawal,2024-01-01,1
9938,1691.25,Withdrawal,2024-01-01,1
8748,850.50,Withdrawal,2024-01-01,1
